# 05 — Segmentasi Citra, Morfologi Matematis, dan Pengukuran Objek

### Dari piksel yang dikelompokkan menuju besaran fisik yang dapat dipertanggungjawabkan

**Pertanyaan utama:** kapan suatu wilayah terang atau gelap boleh disebut satu objek? Mengapa hasil segmentasi yang tampak meyakinkan dapat menghasilkan jumlah objek, luas, dan diameter yang salah? Bagaimana kita mengevaluasi mask ketika pencahayaan, blur, anotasi, dan skala pengukuran tidak sepenuhnya pasti?

Notebook ini menghubungkan teori keputusan berbasis intensitas, statistik histogram, morfologi himpunan, topologi diskret, segmentasi berbasis jarak, geometri objek, dan evaluasi metrologi. Fokus kita bukan hanya menghasilkan gambar biner, melainkan membangun **rantai pengukuran** dari citra ke estimasi kuantitas objek, lengkap dengan asumsi, *ground truth*, dan batas klaimnya.

**Prasyarat:** Notebook 01 (sampling, kuantisasi, sensor), 02 (histogram dan transformasi), 03 (filtering, blur, noise), 04 (gradien, tepi, ketidakpastian posisi); Python/NumPy, probabilitas dasar, serta operasi himpunan. **Komputasi:** CPU; tanpa GPU, kredensial API, atau unduhan dataset tambahan.

**Praktikum:** [Jalankan melalui Google Colab setelah notebook di-commit](https://colab.research.google.com/github/iwanconysetiadi/ImageProcessing/blob/main/notebooks/05_segmentasi.ipynb).

> **Provenance:** contoh `coins` disediakan oleh `scikit-image` dan digunakan untuk pengantar eksploratif saja. Seluruh mask acuan, skala panjang, nilai intensitas, akuisisi kalibrasi, dan metrik pada studi kasus merupakan **simulasi sintetis**. Tidak ada bukti validasi alat ukur maupun inspeksi lapangan di notebook ini.

## Peta kompetensi dan pertanyaan pemandu

Setelah menyelesaikan materi, pembaca diharapkan mampu:

1. Membedakan segmentasi semantik, instance, dan segmentasi untuk pengukuran fisik.
2. Menurunkan ambang Otsu dari varian intra- dan antarkelas, termasuk kondisi gagalnya.
3. Menjelaskan perbedaan *global threshold*, *local threshold*, dan koreksi iluminasi radiometrik.
4. Menurunkan erosi, dilasi, opening, dan closing menggunakan operasi himpunan serta menilai efek ukuran elemen struktur.
5. Menganalisis konektivitas 4/8, komponen terhubung, topologi, dan risiko menggabungkan dua benda menjadi satu label.
6. Menggunakan transformasi jarak dan watershed untuk memisahkan komponen berdekatan, termasuk kasus oversegmentation.
7. Mengubah pengukuran luas, centroid, dan diameter dari piksel menjadi satuan fisik dengan kalibrasi yang dinyatakan eksplisit.
8. Membedakan Dice/IoU tingkat piksel dan precision–recall/F1 tingkat objek, serta memahami mekanisme *matching* antar-instance.
9. Menilai ketidakpastian akibat posisi batas, variasi anotasi, skala, dan replikasi eksperimen.
10. Membangun protokol validasi terpisah dari pengujian akhir, menentukan parameter tanpa *data leakage*, serta melaporkan kegagalan secara ilmiah.

**Pertanyaan pemandu:** Apakah mask yang sedikit lebih besar selalu lebih baik? Bagaimana satu sambungan tipis antara dua partikel mengubah jumlah objek tanpa banyak mengubah IoU? Apakah ketidakpastian rata-rata luas bisa dihitung dengan memperlakukan setiap piksel sebagai observasi independen?

## 1. Kontrak data, koordinat, dan interpretasi label

Kita menuliskan citra sebagai $I[m,n]$ dengan indeks baris $m$ (ke bawah) dan kolom $n$ (ke kanan). Pada segmentasi biner, $M[m,n]\in\{0,1\}$ menunjukkan kelas objek. Untuk segmentasi *instance*, $L[m,n]\in\{0,1,\ldots,K\}$ dengan nol sebagai latar dan setiap bilangan positif merepresentasikan identitas objek berbeda.

**Tiga kategori hasil yang tidak boleh dicampur:**

- **Semantic mask:** piksel mana yang termasuk kelas tertentu, tanpa membedakan objek sekelas.
- **Instance mask:** piksel mana milik masing-masing objek fisik.
- **Measurand:** besaran yang sebenarnya hendak diukur, misalnya jumlah partikel $N$, luas proyeksi $A$, atau diameter ekuivalen $d_\mathrm{eq}$.

Dua mask dapat mempunyai IoU hampir sama tetapi prediksi jumlah objek berbeda. Sebaliknya, jumlah komponen digital belum tentu identik dengan jumlah benda nyata bila objek saling bersentuhan atau terhalang.

Gunakan citra `float` agar skala numerik eksplisit. Nilai `[0,1]` merupakan nilai representasi, **bukan otomatis radiansi atau reflektansi terkalibrasi**. Semua panjang dalam mm nanti memakai *pitch* pada bidang objek yang **diasumsikan**, bukan kalibrasi perangkat nyata.

In [ ]:
import platform, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from skimage import data, filters, morphology, measure, segmentation, feature, draw
from skimage.util import img_as_float
from skimage.color import label2rgb
from IPython.display import display

SEED = 202605
rng = np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi':110,'font.size':10,'axes.grid':False})

def lihat(ax, x, title, *, cmap='gray', vmin=None, vmax=None):
    ax.imshow(x, cmap=cmap, vmin=vmin, vmax=vmax, interpolation='nearest')
    ax.set_title(title)
    ax.axis('off')

coins = img_as_float(data.coins())
print('Python:',platform.python_version(),'| shape citra contoh:',coins.shape,
      '| rentang:',np.round([coins.min(),coins.max()],3))
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
lihat(ax[0],coins,'Citra contoh coins (scikit-image)',vmin=0,vmax=1)
ax[1].hist(coins.ravel(),bins=64,color='0.4')
ax[1].set(title='Histogram bukan peta lokasi',xlabel='Nilai piksel',ylabel='Jumlah piksel')
plt.tight_layout(); plt.show()

## 2. Segmentasi sebagai keputusan statistik

Pada metode ambang, keputusan piksel dapat dinyatakan

$$
\widehat M[m,n]=\mathbf{1}\{I[m,n]<t\},
$$

untuk objek yang secara umum lebih gelap daripada latar. Tanda pertidaksamaan **harus** disesuaikan dengan arah kontras, bukan dianggap tetap. Fungsi indikator $\mathbf 1\{\cdot\}$ bernilai satu bila kondisi terpenuhi.

Secara statistik, bila distribusi intensitas kelas objek $p(I\mid C_1)$ dan latar $p(I\mid C_0)$ diketahui, aturan Bayes dengan biaya kesalahan berbeda dapat ditulis

$$
\frac{p(I\mid C_1)}{p(I\mid C_0)}
> \frac{P(C_0)\,\lambda_{10}}{P(C_1)\,\lambda_{01}},
$$

untuk definisi $\lambda_{10}$ sebagai kerugian memilih objek ketika sebenarnya latar (*false positive*) dan $\lambda_{01}$ sebagai kerugian memilih latar ketika sebenarnya objek (*false negative*). Aturan ini memerlukan asumsi tentang densitas, prior, dan biaya. **Otsu tidak menggunakan label kelas maupun prior biaya eksplisit**; ia memilih ambang berdasarkan struktur histogram.

Jika intensitas dipengaruhi iluminasi spasial, distribusi yang sama dapat berasal dari bagian objek berbeda sehingga satu ambang global tidak cukup. Filter dan histogram pada notebook sebelumnya membantu menerangkan sumber ketidakstabilan tersebut.

In [ ]:
# Bandingkan ambang global dengan segmentasi terang/gelap; ini eksplorasi, bukan ground truth.
t_coins=filters.threshold_otsu(coins)
dark=coins<t_coins
bright=coins>t_coins
fig,ax=plt.subplots(1,3,figsize=(12,3.7))
lihat(ax[0],coins,'Intensitas asli',vmin=0,vmax=1)
lihat(ax[1],dark,f'Asumsi objek gelap (t={t_coins:.3f})')
lihat(ax[2],bright,'Asumsi objek terang')
plt.tight_layout();plt.show()
print('Luas foreground asumsi gelap:',dark.mean(),'| asumsi terang:',bright.mean())

## 3. Otsu: penurunan dari histogram dan varians antarkelas

Untuk intensitas diskret $i\in\{0,\ldots,L-1\}$ dengan histogram ternormalisasi $p_i$ ($\sum_i p_i=1$), ambang $t$ membagi dua kelas:

$$
\omega_0(t)=\sum_{i\le t}p_i,\qquad
\omega_1(t)=1-\omega_0(t).
$$

Rata-rata kelas adalah $\mu_0$ dan $\mu_1$; rata-rata keseluruhan $\mu_T=\omega_0\mu_0+\omega_1\mu_1$. Varians total memenuhi *law of total variance*:

$$
\sigma_T^2=\underbrace{\omega_0\sigma_0^2+\omega_1\sigma_1^2}_{\sigma_W^2(t)}+
\underbrace{\omega_0\omega_1(\mu_0-\mu_1)^2}_{\sigma_B^2(t)}.
$$

Karena $\sigma_T^2$ tidak bergantung pada $t$, meminimalkan varians dalam kelas $\sigma_W^2$ setara dengan memaksimalkan varians antarkelas $\sigma_B^2$. Bila $\omega_0=0$ atau $\omega_1=0$, pemisahan kelas tidak terdefinisi, sehingga kandidat itu dikecualikan.

**Konsekuensi:** Otsu efektif bila histogram relatif terbagi menjadi dua populasi yang cukup berbeda. Ia dapat gagal jika foreground jarang, latar bergradien, banyak tekstur, atau distribusi intensitas saling tumpang tindih. Ambang optimal terhadap varians histogram juga belum tentu optimal untuk kesalahan luas, jumlah objek, ataupun dampak keputusan yang tidak simetris.

In [ ]:
# Implementasi Otsu dari definisi (menggunakan 256 bin) untuk memeriksa library.
hist, edges=np.histogram(coins.ravel(),bins=256,range=(0,1))
p=hist/hist.sum()
c=(edges[:-1]+edges[1:])/2
w0=np.cumsum(p); w1=1-w0
mu_partial=np.cumsum(p*c); muT=mu_partial[-1]
valid=(w0>1e-12)&(w1>1e-12)
between=np.full_like(w0,np.nan)
between[valid]=(muT*w0[valid]-mu_partial[valid])**2/(w0[valid]*w1[valid])
t_manual=c[np.nanargmax(between)]
print('Otsu manual (bin 256):',round(t_manual,4),'| scikit-image:',round(t_coins,4))
fig,ax=plt.subplots(1,2,figsize=(11,3.4))
ax[0].plot(c,p,label='Probabilitas intensitas');ax[0].axvline(t_manual,color='k',ls='--')
ax[0].set(xlabel='Nilai piksel',ylabel='Probabilitas',title='Histogram ternormalisasi')
ax[1].plot(c,between,lw=1.5)
ax[1].axvline(t_manual,color='k',ls='--')
ax[1].set(xlabel='Ambang t',ylabel='Varians antarkelas',title='Fungsi objektif Otsu')
plt.tight_layout();plt.show()
assert abs(t_manual-t_coins)<1/256+1e-3

## 4. Ambang lokal dan perubahan iluminasi

Untuk citra hasil akuisisi sederhana, intensitas teramati dapat didekati sebagai

$$
I[m,n]=d[m,n]+g[m,n]\,\ell[m,n]\,r[m,n]+\eta[m,n],
$$

dengan $d$ adalah *dark offset*, $g$ gain, $\ell$ pola iluminasi, $r$ reflektansi efektif, dan $\eta$ komponen noise. Produk $g\ell r$ berarti dua objek dengan reflektansi identik dapat menghasilkan kode intensitas berbeda ketika iluminasi berubah.

**Ambang lokal** mengizinkan nilai ambang berubah sebagai fungsi posisi, misalnya $t[m,n]=\mu_{\mathcal N}[m,n]-c$, dengan $\mu_{\mathcal N}$ rata-rata lingkungan piksel. Kekurangannya: tekstur dan objek besar dapat menggeser estimasi latar itu sendiri.

**Koreksi flat-field** (bila tersedia citra kalibrasi yang cocok) berusaha memperkirakan efek respons medan sebelum melakukan keputusan. Ini berbeda dari mempercantik gambar: penggunaan *dark frame* dan *flat frame* adalah bagian dari model sensor dan wajib didokumentasikan, termasuk keterbatasannya.

In [ ]:
# Citra sintetis: dua objek gelap yang identik tetapi mendapat iluminasi berbeda.
H0,W0=140,210
yy,xx=np.mgrid[:H0,:W0]
illum_demo=.57+.40*(xx/(W0-1))
mask_demo=((xx-52)**2+(yy-68)**2<18**2)|((xx-161)**2+(yy-68)**2<18**2)
im_demo=ndi.gaussian_filter(illum_demo*(1-.45*mask_demo),sigma=1.0)
im_demo=np.clip(im_demo+rng.normal(0,.014,im_demo.shape),0,1)
t_global=filters.threshold_otsu(im_demo)
mask_global=im_demo<t_global
mask_local=im_demo<filters.threshold_local(im_demo,block_size=51,offset=.085)
fig,ax=plt.subplots(1,4,figsize=(13,3.0))
for a,x,t in zip(ax,[im_demo,mask_demo,mask_global,mask_local],
               ['Intensitas tidak seragam','Ground truth','Otsu global','Ambang lokal']):
    lihat(a,x,t,vmin=0 if x is im_demo else None,vmax=1 if x is im_demo else None)
plt.tight_layout();plt.show()
for name,pred in [('Global',mask_global),('Lokal',mask_local)]:
    tp=np.count_nonzero(pred&mask_demo)
    dice=2*tp/(np.count_nonzero(pred)+np.count_nonzero(mask_demo))
    print(f'Dice {name}: {dice:.3f}')

## 5. Morfologi matematis: operasi pada himpunan, bukan sekadar membersihkan noise

Anggap objek biner sebagai himpunan koordinat $A\subset\mathbb Z^2$ dan elemen struktur sebagai $B$. Dilasi dan erosi didefinisikan melalui translasi himpunan:

$$
A\oplus B=\{a+b\mid a\in A,\ b\in B\},
\qquad
A\ominus B=\{x\mid x+B\subseteq A\}.
$$

Dengan elemen struktur yang memuat titik asal, erosi mengecilkan objek dan dilasi memperbesar objek. Komposisinya menghasilkan **opening** dan **closing**:

$$
A\circ B=(A\ominus B)\oplus B,
\qquad
A\bullet B=(A\oplus B)\ominus B.
$$

Opening cenderung menghilangkan struktur foreground kecil atau penghubung sempit; closing cenderung mengisi celah kecil atau menghubungkan bagian yang berdekatan. Akan tetapi, tidak ada jaminan bahwa keduanya memperbaiki ukuran objek secara fisik—batas objek dapat bergeser.

Sifat berguna pada domain tanpa artefak batas, dengan definisi structuring element konsisten: *opening* dan *closing* monoton serta idempoten, yaitu $(A\circ B)\circ B=A\circ B$ dan $(A\bullet B)\bullet B=A\bullet B$. Efek dekat batas citra memerlukan perhatian pada *padding*.

In [ ]:
# Morfologi biner pada objek ber-noise dan lubang sintetis.
mask0=np.zeros((125,190),dtype=bool)
rr,cc=draw.disk((64,56),30,shape=mask0.shape);mask0[rr,cc]=True
rr,cc=draw.disk((64,127),26,shape=mask0.shape);mask0[rr,cc]=True
mask0[61:68,84:103]=True          # sambungan sempit yang bersifat artifisial
mask0[60:66,55:61]=False        # lubang
mask0[12,19]=True;mask0[115,174]=True
se=morphology.disk(3)
variants={'Asli':mask0,'Erosi':morphology.erosion(mask0,se),
          'Dilasi':morphology.dilation(mask0,se),
          'Opening':morphology.opening(mask0,se),
          'Closing':morphology.closing(mask0,se)}
fig,ax=plt.subplots(1,5,figsize=(14,3.2))
for a,(name,m) in zip(ax,variants.items()):
    lihat(a,m,f'{name}\nluas={m.sum()} piksel')
plt.tight_layout();plt.show()
print('Opening idempoten:',np.array_equal(
    morphology.opening(variants['Opening'],se),variants['Opening']))

## 6. Morfologi aras abu-abu dan elemen struktur dalam satuan fisik

Untuk citra intensitas $I$, morfologi juga dapat dinyatakan sebagai minimum/maksimum lokal. Untuk elemen struktur datar $B$,

$$
(\varepsilon_B I)(x)=\min_{b\in B} I(x+b),
\qquad
(\delta_B I)(x)=\max_{b\in B} I(x-b),
$$

tergantung konvensi refleksi elemen struktur. Karena itu, morfologi aras abu-abu tidak identik dengan memproses mask biner hasil thresholding. *White top-hat* sering mengambil selisih citra dengan opening, sedangkan *black top-hat* mengambil selisih closing dengan citra untuk menonjolkan fitur gelap.

**Satuan menentukan ukuran kernel.** Jika ukuran piksel pada bidang objek $p_x$ dan $p_y$ berbeda, cakram radius $r$ mm secara fisik menjadi elemen struktur elips pada grid: $(\Delta n\,p_x)^2+(\Delta m\,p_y)^2\le r^2$. Memakai cakram 5 piksel pada citra anisotropik **tidak** sama dengan cakram radius tetap dalam mm.

In [ ]:
# Bandingkan white/black top-hat serta kernel dengan satuan panjang fisik.
y,x=np.mgrid[:110,:170]
gray=.36+.22*(x/170)+.10*np.sin(y/25)
gray+=.20*np.exp(-((x-49)**2+(y-48)**2)/65)
gray-=.18*np.exp(-((x-125)**2+(y-58)**2)/85)
gray=np.clip(gray,0,1)
white=morphology.white_tophat(gray,footprint=morphology.disk(10))
black=morphology.black_tophat(gray,footprint=morphology.disk(10))
p_x,p_y=.038,.058  # mm/piksel; ASUMSI illustratif.
r_mm=.24
rad_x=int(np.ceil(r_mm/p_x));rad_y=int(np.ceil(r_mm/p_y))
Y,X=np.mgrid[-rad_y:rad_y+1,-rad_x:rad_x+1]
se_phys=((X*p_x)**2+(Y*p_y)**2)<=r_mm**2
fig,ax=plt.subplots(1,4,figsize=(13,3))
for a,img,title in zip(ax,[gray,white,black,se_phys],
                       ['Citra sintetis','White top-hat','Black top-hat','SE 0.24 mm (anisotropik)']):
    lihat(a,img,title)
plt.tight_layout();plt.show()
print('Elemen struktur:',se_phys.shape,'piksel; luas:',se_phys.sum(),'piksel')

## 7. Komponen terhubung, definisi tetangga, dan topologi digital

Labeling pada mask biner menuntut definisi konektivitas. Pada grid dua dimensi, **4-konektivitas** mencakup tetangga utara, selatan, barat, dan timur; **8-konektivitas** juga mencakup diagonal. Dua piksel yang hanya bersentuhan diagonal merupakan satu komponen dalam 8-konektivitas, tetapi dua komponen dalam 4-konektivitas.

Pilih konvensi foreground dan background secara konsisten untuk mencegah ambiguitas digital-topologi. Menggunakan konektivitas yang sama untuk keduanya dapat memunculkan paradoks geometris terkait keterhubungan batas.

Komponen terhubung hanya mengukur **kelompok piksel**, bukan jumlah objek fisik. Jika dua objek saling bersinggungan akibat PSF atau closing, keduanya mungkin dilabeli sebagai satu komponen. Jika satu benda tertutup bayangan/berlubang, ia bisa terpecah. Topologi bukan hal dekoratif ketika keluaran digunakan untuk menghitung partikel atau pori.

In [ ]:
# Contoh paling sederhana: dua piksel diagonal.
demo_conn=np.array([[1,0,0,0], [0,1,0,0], [0,0,0,0], [0,0,1,1]],dtype=bool)
lab4=measure.label(demo_conn,connectivity=1)
lab8=measure.label(demo_conn,connectivity=2)
fig,ax=plt.subplots(1,3,figsize=(9,3))
lihat(ax[0],demo_conn,'Mask awal')
lihat(ax[1],lab4,f'4 tetangga: {lab4.max()} region',cmap='nipy_spectral')
lihat(ax[2],lab8,f'8 tetangga: {lab8.max()} region',cmap='nipy_spectral')
plt.tight_layout();plt.show()
assert lab4.max()==3 and lab8.max()==2

## 8. Transformasi jarak dan watershed untuk memisahkan objek bersinggungan

Untuk mask objek $M$, *Euclidean distance transform* menghitung jarak setiap piksel objek ke latar terdekat:

$$
D(x)=\min_{z\,:\,M(z)=0}\|x-z\|_2.
$$

Pusat objek cembung cenderung membentuk maksimum lokal pada peta $D$. *Marker-controlled watershed* membagi daerah foreground menjadi cekungan berdasarkan penanda (*markers*), lazimnya menggunakan $-D$ sebagai topografi. Hasilnya sensitif pada *markers*, skala, noise, dan pilihan jarak.

**Kegagalan yang penting:** satu objek memanjang dengan beberapa puncak dapat terpecah (*oversegmentation*); dua objek yang sangat berdekatan dengan satu maksimum dapat menyatu (*undersegmentation*). Segmentasi terbelah yang tampak menarik secara visual tidak otomatis identik dengan instance fisik.

In [ ]:
# Dua cakram hampir bersentuhan untuk memperlihatkan pemisahan instance.
touch=np.zeros((125,180),dtype=bool)
for center in [(62,71),(62,110)]:
    rr,cc=draw.disk(center,21,shape=touch.shape)
    touch[rr,cc]=True
touch=morphology.closing(touch,morphology.disk(1))
dist=ndi.distance_transform_edt(touch)
peaks=feature.peak_local_max(ndi.gaussian_filter(dist,1),min_distance=16,
                             threshold_abs=6,labels=touch,exclude_border=False)
seed_mask=np.zeros_like(touch);seed_mask[tuple(peaks.T)]=True
markers=measure.label(seed_mask)
labels_ws=segmentation.watershed(-dist,markers,mask=touch)
fig,ax=plt.subplots(1,4,figsize=(13,3.2))
lihat(ax[0],touch,'Mask biner')
lihat(ax[1],dist,'Jarak ke latar',cmap='viridis')
lihat(ax[2],markers,'Penanda',cmap='nipy_spectral')
lihat(ax[3],labels_ws,f'Watershed: {labels_ws.max()} instance',cmap='nipy_spectral')
plt.tight_layout();plt.show()
assert labels_ws.max()>=1

## 9. Region properties dan pengukuran geometri

Misalkan satu objek pada mask memiliki $N$ piksel. Untuk piksel persegi panjang pada bidang objek dengan pitch $p_x$ dan $p_y$ (mm/piksel), luas proyeksinya didekati sebagai

$$
\widehat A=N\,p_xp_y\quad\text{(mm}^2\text{)}.
$$

Centroid piksel diperoleh sebagai rata-rata koordinat foreground, misalnya $\bar x=(1/N)\sum_{(m,n)\in A} n p_x$. Diameter lingkaran dengan luas sama (*equivalent diameter*) adalah

$$
\widehat d_{\mathrm{eq}}=2\sqrt{\frac{\widehat A}{\pi}}.
$$

Ini **bukan** diameter Feret, ukuran sumbu mayor, atau ukuran partikel tiga dimensi. Perimeter objek digital lebih peka terhadap sampling, orientasi, definisi estimator, dan *pixelation* daripada luas. Mengukur objek nyata membutuhkan kalibrasi magnifikasi, distorsi lensa, kedalaman, dan geometri proyeksi—bukan hanya memasukkan skala mm/piksel ke rumus.

In [ ]:
# Objek pada citra bawaan: pengukuran masih dalam unit piksel (belum terkalibrasi).
threshold=filters.threshold_otsu(coins)
mask_coins=coins>threshold
mask_coins=morphology.area_opening(mask_coins,area_threshold=100)
regions=measure.regionprops(measure.label(mask_coins))
largest=sorted(regions,key=lambda r:r.area,reverse=True)[:8]
tab=pd.DataFrame([{'label':r.label,'area_px2':r.area,'centroid_y_px':r.centroid[0],
                   'centroid_x_px':r.centroid[1],
                   'equivalent_diameter_px':r.equivalent_diameter_area,
                   'solidity':r.solidity} for r in largest])
display(tab.round(2))
fig,ax=plt.subplots(1,2,figsize=(11,4))
lihat(ax[0],coins,'Citra contoh',vmin=0,vmax=1)
lihat(ax[1],measure.label(mask_coins),'Komponen hasil threshold',cmap='nipy_spectral')
plt.tight_layout();plt.show()
print('Catatan: labeling Otsu di atas BUKAN label koin ground truth.')

## 10. Ketidakpastian luas: dari batas piksel ke kalibrasi ukuran

Untuk luas $A=Np_xp_y$, bila kita memperlakukan $N$, $p_x$, dan $p_y$ sebagai input acak dengan ketidakpastian standar kecil, pendekatan orde pertama memberi

$$
 u_A^2\approx (p_xp_y)^2u_N^2+(Np_y)^2u_{p_x}^2+(Np_x)^2u_{p_y}^2
 +2\sum_{i<j}\frac{\partial A}{\partial q_i}\frac{\partial A}{\partial q_j}\operatorname{Cov}(q_i,q_j),
$$

untuk $\mathbf q=(N,p_x,p_y)$. **Jika kalibrasi $p_x$ dan $p_y$ berkorelasi, suku kovarians tidak boleh dibuang.** Karena piksel foreground bukan replikasi pengukuran independen, $u_N$ tidak dapat diasumsikan otomatis sama dengan $\sqrt N$; ia bergantung pada galat segmentasi, resolusi, threshold, blur, dan variasi anotasi.

Penggunaan $N$ bilangan bulat tidak menghilangkan ketidakpastian: perubahan threshold kecil dapat memindahkan banyak piksel pada batas objek. Sebaliknya, memperlakukan seluruh piksel sebagai sampel statistik independen akan menghasilkan interval terlalu sempit.

In [ ]:
# Contoh perhitungan propagasi dengan input ilustratif, bukan ketidakpastian alat.
N=2350; ux_N=85
px=.040;py=.044;u_px=.0005;u_py=.0006
rho_xy=.60  # korelasi kalibrasi pitch yang DIASUMSIKAN
A=N*px*py
grad=np.array([px*py,N*py,N*px])
Cov=np.array([[ux_N**2,0,0],
              [0,u_px**2,rho_xy*u_px*u_py],
              [0,rho_xy*u_px*u_py,u_py**2]])
u_A=np.sqrt(grad@Cov@grad)
uncorr=np.sqrt(np.sum(grad**2*np.diag(Cov)))
print(f'Area nominal: {A:.3f} mm²')
print(f'u(A) dengan korelasi pitch: {u_A:.3f} mm²; jika korelasi diabaikan: {uncorr:.3f} mm²')
assert u_A>0 and A>0

## 11. Metrik tingkat piksel dan tingkat objek tidak ekuivalen

Untuk segmentasi biner, definisikan $TP$, $FP$, dan $FN$ sebagai jumlah piksel berdasarkan perbandingan dengan mask acuan. Dua metrik penting:

$$
\operatorname{IoU}=\frac{TP}{TP+FP+FN},\qquad
\operatorname{Dice}=\frac{2TP}{2TP+FP+FN}.
$$

Untuk mask biner yang sama dan penyebut non-nol, $\mathrm{Dice}=2\mathrm{IoU}/(1+\mathrm{IoU})$. Keduanya tidak menghukum *split* dan *merge* instance dengan cara yang sama seperti evaluasi tingkat objek.

Dalam evaluasi objek, kita cocokkan prediksi dan instance acuan secara **satu-ke-satu** menggunakan IoU antar-instance, kemudian tetapkan ambang kecocokan (misalnya $\tau=0.5$). Dari $TP_{obj}$, $FP_{obj}$, dan $FN_{obj}$, didapat precision, recall, dan F1 tingkat objek. Proses *matching* dan ambang harus didokumentasikan; mengganti ambang setelah melihat hasil test merupakan evaluasi yang bias.

Jika proyek membutuhkan ketepatan ukuran, tambahkan metrik $|\widehat A-A|$, kesalahan jumlah, dan kesalahan posisi. Metrik yang bagus untuk segmentasi piksel belum tentu sesuai untuk *metrology*.

In [ ]:
# Identitas Dice–IoU dan contoh penyatuan dua objek yang membingungkan hitungan.
def pixel_metrics(pred,true):
    pred=np.asarray(pred,dtype=bool);true=np.asarray(true,dtype=bool)
    tp=np.count_nonzero(pred&true);fp=np.count_nonzero(pred&~true);fn=np.count_nonzero(~pred&true)
    iou=tp/(tp+fp+fn) if (tp+fp+fn) else 1.0
    dice=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 1.0
    return dict(iou=iou,dice=dice,precision=tp/(tp+fp) if tp+fp else 0.,
                recall=tp/(tp+fn) if tp+fn else 0.)
truth_example=np.zeros((100,140),dtype=int)
for ident,(cy,cx) in enumerate([(51,49),(51,91)],1):
    rr,cc=draw.disk((cy,cx),19,shape=truth_example.shape)
    truth_example[rr,cc]=ident
merged=truth_example>0
merged[48:55,67:73]=True  # sambungan sempit: dua instance kini bisa menjadi satu.
lab_merged=measure.label(merged,connectivity=2)
pix=pixel_metrics(merged,truth_example>0)
print('IoU piksel=',round(pix['iou'],4),'| Dice=',round(pix['dice'],4),
      '| objek benar=',truth_example.max(),'| label prediksi=',lab_merged.max())
assert np.isclose(pix['dice'],2*pix['iou']/(1+pix['iou']))
fig,ax=plt.subplots(1,2,figsize=(8,3))
lihat(ax[0],truth_example,'Ground truth 2 objek',cmap='nipy_spectral')
lihat(ax[1],lab_merged,'Sambungan menyebabkan merge',cmap='nipy_spectral')
plt.tight_layout();plt.show()

## 12. Matching objek, oversegmentation, dan undersegmentation

Untuk dua peta label $L$ (acuan) dan $\widehat L$ (prediksi), bentuk matriks $J_{ij}=\operatorname{IoU}(L=i,\widehat L=j)$. Kita memakai penugasan optimal satu-ke-satu berdasarkan jumlah IoU melalui *Hungarian algorithm*. Pasangan yang IoU-nya kurang dari $\tau$ dihitung tidak cocok; sisanya menentukan $TP_{obj}$, sedangkan label tak cocok menentukan $FP_{obj}$ dan $FN_{obj}$.

Penting: memaksimalkan jumlah IoU kemudian menerapkan ambang bukan identik dengan setiap definisi deteksi objek yang mungkin, terutama jika matching sangat ambigu; ini adalah **protokol yang ditetapkan dalam notebook**. Untuk kasus sulit, jelaskan apakah yang dimaksimalkan jumlah pasangan valid, jumlah IoU, atau skor lain.

Sebuah objek yang terpecah dua menghasilkan satu prediksi tambahan dan potensi *false positive*. Dua benda yang menyatu menghasilkan *false negative*. Kedua kasus dapat memiliki IoU piksel yang cukup besar meskipun jumlah objek salah.

In [ ]:
def instance_metrics(pred_labels,true_labels,iou_threshold=.5):
    pred_labels=np.asarray(pred_labels);true_labels=np.asarray(true_labels)
    ids_t=np.unique(true_labels);ids_t=ids_t[ids_t>0]
    ids_p=np.unique(pred_labels);ids_p=ids_p[ids_p>0]
    mat=np.zeros((len(ids_t),len(ids_p)),float)
    true_masks=[true_labels==i for i in ids_t]
    pred_masks=[pred_labels==j for j in ids_p]
    for i,mi in enumerate(true_masks):
        for j,mj in enumerate(pred_masks):
            inter=np.count_nonzero(mi & mj)
            union=np.count_nonzero(mi | mj)
            mat[i,j]=inter/union if union else 0.
    pairs=[]
    if mat.size:
        ir,ic=linear_sum_assignment(1-mat)
        pairs=[(int(ids_t[i]),int(ids_p[j]),float(mat[i,j]))
               for i,j in zip(ir,ic) if mat[i,j]>=iou_threshold]
    tp=len(pairs);fp=len(ids_p)-tp;fn=len(ids_t)-tp
    prec=tp/(tp+fp) if (tp+fp) else 0.
    rec=tp/(tp+fn) if (tp+fn) else 0.
    f1=2*prec*rec/(prec+rec) if (prec+rec) else 0.
    return {'tp_obj':tp,'fp_obj':fp,'fn_obj':fn,
            'precision_obj':prec,'recall_obj':rec,'f1_obj':f1,
            'pairs':pairs,'iou_matrix':mat}

a=instance_metrics(lab_merged,truth_example,.5)
print('TP,FP,FN objek =',a['tp_obj'],a['fp_obj'],a['fn_obj'],
      '| F1 objek=',a['f1_obj'])
print('Matriks IoU antar-instance:',np.round(a['iou_matrix'],3))
assert a['tp_obj']==0 and a['fn_obj']==2 and a['fp_obj']==1

## 13. Variabilitas anotasi dan batas definisi ground truth

Anotasi manusia bukan pengukuran sempurna yang bebas asumsi. Pada batas yang blur, dua anotator dapat memilih batas berbeda walaupun keduanya konsisten. Ketidakpastian ini perlu dicatat sebagai variasi prosedural, **bukan otomatis sebagai noise sensor**.

Beberapa sumber variasi anotasi ialah pilihan aturan batas, resolusi tampilan, kualitas monitor, pengetahuan ahli, dan keputusan tentang apakah objek kecil diikutkan. Jika satu anotasi tunggal dipakai sebagai acuan, nyatakan siapa yang membuatnya dan bagaimana sengketa label diselesaikan.

Untuk proyek nyata, ukur kesepakatan antaranotator (misalnya Dice antarmask, variasi diameter, atau kesepakatan tingkat objek), sertakan protokol anotasi, dan pertimbangkan evaluasi terhadap beberapa anotator. Simulasi berikut hanya menggambarkan pengaruh perubahan batas; **tidak dapat menggantikan studi inter-annotator nyata**.

In [ ]:
# Variasi anotasi sintetis: satu piksel erosi/dilasi di batas.
ann0=truth_example>0
ann_in=morphology.erosion(ann0,morphology.disk(1))
ann_out=morphology.dilation(ann0,morphology.disk(1))
print('Luas mask referensi, inward, outward (piksel):',
      [int(z.sum()) for z in [ann0,ann_in,ann_out]])
print('Dice inward versus referensi:',round(pixel_metrics(ann_in,ann0)['dice'],3),
      '| Dice outward:',round(pixel_metrics(ann_out,ann0)['dice'],3))
fig,ax=plt.subplots(1,3,figsize=(9,3))
for a,z,t in zip(ax,[ann_in,ann0,ann_out],['Batas ke dalam','Referensi','Batas ke luar']):lihat(a,z,t)
plt.tight_layout();plt.show()

## 14. Studi kasus integratif — Penghitungan dan pengukuran partikel pada citra sensor

**Masalah engineering.** Sebuah sistem kamera memotret sekumpulan objek gelap di atas permukaan yang lebih terang. Tujuannya adalah memperkirakan **jumlah objek**, **luas total proyeksi**, dan **luas per instance** dalam mm². Pencahayaan berubah antarspesimen; respons sensor melibatkan blur, noise foton, read noise, dan kuantisasi. Untuk pendekatan tertentu, tersedia *dark frame* dan *flat frame* yang diasumsikan diambil pada kondisi akuisisi sama.

**Definisi hasil yang dinilai:**

- IoU/Dice mask foreground terhadap mask sintetis benar.
- F1 *instance* dengan matching IoU $\ge 0.5$.
- Galat absolut jumlah $|\widehat K-K|$ per spesimen.
- Galat absolut luas proyeksi total $|\widehat A-A|$ (mm²) per spesimen.
- Waktu eksekusi pada mesin yang digunakan, hanya sebagai ilustrasi efisiensi.

**Unit replikasi statistik ialah spesimen**, bukan ratusan ribu piksel di dalam satu gambar. Data **validasi** digunakan untuk memilih parameter; data **uji** dibiarkan tertutup sampai parameter diputuskan. Data uji sintetis tidak disetarakan dengan validasi lapangan.

### 14A. Model generatif, kalibrasi, dan label acuan yang dapat diaudit

Kita bangun peta instance $L_\star$ dari elips terrotasi yang tidak saling tumpang tindih. Peta reflektansi efektif $r$ dikonvolusi oleh PSF Gaussian dan dikalikan iluminasi spasial $\ell$ serta gain $g$. Kemudian jumlah elektron/foton efektif dibangkitkan menggunakan Poisson dan noise baca Gaussian:

$$
I=Q_{12}\!\left[d+\frac{\operatorname{Poisson}(N_e\,g\ell\,(h*r))}{N_e}+\epsilon_r\right],
\qquad \epsilon_r\sim\mathcal N(0,\sigma_r^2),
$$

untuk kuantisasi $Q_{12}$ sebesar 12-bit pada rentang `[0,1]`. Model ini adalah idealisasi, bukan karakterisasi spesifik kamera. Semakin rendah iluminasi, gangguan relatif dapat meningkat.

*Flat frame* dibuat dari bidang dengan reflektansi konstan dan iluminasi yang sama, tetapi dengan noise kalibrasi berbeda. Koreksi digunakan sebagai

$$
\widehat r\;\propto\;\frac{I-D}{F-D+\varepsilon},
$$

dengan $D$ *dark frame*, $F$ *flat frame*, dan $\varepsilon$ proteksi numerik. Koreksi ini tidak sepenuhnya menghapus blur, noise, perubahan reflektansi objek, atau kesalahan pengambilan kalibrasi. Informasi kalibrasi **disediakan oleh simulator secara eksplisit**, bukan dipelajari dari label benar.

In [ ]:
PITCH_X_MM=.042   # mm/piksel, asumsi simulasi
PITCH_Y_MM=.046   # mm/piksel, asumsi simulasi
H,W=164,196

def simulate_specimen(seed,*,shift=False):
    r=np.random.default_rng(int(seed))
    yy,xx=np.mgrid[:H,:W]
    labels=np.zeros((H,W),dtype=np.int32)
    n_target=int(r.integers(8,12))
    instances=[]
    for ident in range(1,n_target+1):
        for attempt in range(250):
            cy=int(r.integers(17,H-17));cx=int(r.integers(17,W-17))
            ry=int(r.integers(7,12));rx=int(r.integers(7,14))
            rr,cc=draw.ellipse(cy,cx,ry,rx,rotation=float(r.uniform(-.9,.9)),shape=(H,W))
            # Tidak ada dua label pada piksel sama. Objek tetap dapat berdekatan.
            if np.count_nonzero(labels[rr,cc])==0:
                labels[rr,cc]=ident
                instances.append(ident)
                break
    # Berikan nomor label rapat tanpa lubang identitas.
    unique=np.unique(labels);unique=unique[unique>0]
    remap=np.zeros_like(labels)
    for k,old in enumerate(unique,1):remap[labels==old]=k
    labels=remap
    # Iluminasi bervariasi per spesimen; pergeseran domain sengaja lebih kuat.
    amp=float(r.uniform(.15,.36) if not shift else r.uniform(.38,.56))
    orientation=float(r.uniform(-1,1))
    X=xx/(W-1)-.5;Y=yy/(H-1)-.5
    illum=.88+amp*(orientation*X+np.sqrt(max(0,1-orientation**2))*Y)
    illum+=.11*np.sin(2*np.pi*(X*r.uniform(.4,.8)+Y*.3)+float(r.uniform(0,6)))
    illum=np.clip(illum,.43,1.24)
    reflectance=np.ones((H,W),float)*.86
    for k in unique:
        reflectance[labels==k]=float(r.uniform(.36,.49))
    reflectance+=ndi.gaussian_filter(r.normal(0,.018,(H,W)),sigma=2.2)
    reflectance=np.clip(reflectance,.25,.97)
    blur=ndi.gaussian_filter(reflectance,sigma=(1.5 if shift else 1.05))
    dark_offset=.026
    gain=.81
    electrons=(480 if shift else 1250)
    read_sd=(.025 if shift else .012)
    def acquisition(source,noise_sd):
        lam=np.maximum(gain*illum*source,0)
        sig=r.poisson(lam*electrons)/electrons
        x=dark_offset+sig+r.normal(0,noise_sd,(H,W))
        return np.rint(np.clip(x,0,1)*4095)/4095
    raw=acquisition(blur,read_sd)
    flat=acquisition(np.full((H,W),.86),read_sd*.60)
    dark=np.rint(np.clip(dark_offset+r.normal(0,.0025,(H,W)),0,1)*4095)/4095
    return {'raw':raw,'flat':flat,'dark':dark,'labels':labels,
            'seed':int(seed),'domain':'shift' if shift else 'nominal'}

V_SEEDS=list(range(SEED+1000,SEED+1008))
T_SEEDS=list(range(SEED+5000,SEED+5012))
assert set(V_SEEDS).isdisjoint(T_SEEDS)
val=[simulate_specimen(s) for s in V_SEEDS]
test=[simulate_specimen(s) for s in T_SEEDS]
print('Validasi:',len(val),'spesimen; uji:',len(test),'spesimen')
print('Jumlah objek contoh:',np.unique(val[0]['labels'])[1:].size)
s=val[0]
fig,ax=plt.subplots(1,4,figsize=(14,3.3))
lihat(ax[0],s['raw'],'Citra hasil simulasi',vmin=0,vmax=1)
lihat(ax[1],s['flat'],'Flat frame sintetis',vmin=0,vmax=1)
lihat(ax[2],s['dark'],'Dark frame sintetis',vmin=0,vmax=.08)
lihat(ax[3],s['labels'],'Instance ground truth',cmap='nipy_spectral')
plt.tight_layout();plt.show()

### 14B. Koreksi radiometrik sebagai bagian dari pengukuran

Koreksi *dark–flat* hanya dapat digunakan bila kondisi kalibrasi cocok dengan data pengujian: eksposur, gain, fokus, respons spektral, serta perubahan iluminasi harus dikendalikan. Pada simulator, keadaan tersebut direkayasa agar cocok, sehingga **keunggulan metode koreksi tidak boleh digeneralisasi ke setiap kamera**.

Perhatikan bahwa rasio dapat memperbesar noise ketika penyebut $F-D$ kecil. Proteksi dengan $\varepsilon$ menghindari pembagian nol, tetapi tidak menghilangkan bias dan ketidakpastian kalibrasi. Karena itu kita juga menampilkan peta residual dan distribusi intensitas, bukan hanya citra yang sudah dinormalisasi.

In [ ]:
def flat_correct(sample):
    denom=sample['flat']-sample['dark']
    corrected=.86*(sample['raw']-sample['dark'])/np.maximum(denom,.025)
    return np.clip(corrected,0,1.3)

cor=flat_correct(s)
valid_bg=(s['labels']==0)
fig,ax=plt.subplots(1,3,figsize=(11,3.2))
lihat(ax[0],s['raw'],'Raw',vmin=0,vmax=1)
lihat(ax[1],cor,'Dark–flat corrected',vmin=0,vmax=1)
ax[2].hist(s['raw'][valid_bg],bins=50,alpha=.6,label='Raw (latar)')
ax[2].hist(cor[valid_bg],bins=50,alpha=.6,label='Corrected (latar)')
ax[2].legend(fontsize=8)
ax[2].set(title='Distribusi latar menurut label acuan\n(hanya untuk analisis)',xlabel='Nilai intensitas',ylabel='Frekuensi')
plt.tight_layout();plt.show()
print('Simpangan baku latar raw:',round(float(s['raw'][valid_bg].std()),4),
      '| corrected:',round(float(cor[valid_bg].std()),4))

### 14C. Pipeline, pilihan hyperparameter, dan perbandingan yang adil

Kita bandingkan lima pipeline yang **didefinisikan sebelumnya**:

1. **Raw Otsu** — threshold global citra mentah.
2. **Local** — threshold lokal pada citra mentah dengan offset tetap.
3. **Flat Otsu** — koreksi dark–flat diikuti Otsu.
4. **Flat Fixed** — koreksi dark–flat dan threshold numerik yang diseleksi dari validation set.
5. **Flat + Watershed** — seperti keempat, dengan pemisahan instance menggunakan penanda transformasi jarak.

Semua pipeline menggunakan pembersihan objek kecil dan label komponen (8-konektivitas). Beberapa pipeline dengan *postprocessing* mempunyai kebutuhan tuning berbeda; angka final menilai **pipeline lengkap sesuai protokol**, bukan membuktikan bahwa satu operator secara universal lebih baik.

Parameter threshold dan minimum area dipilih dengan mengoptimalkan **F1 tingkat objek pada data validasi**. IoU tingkat piksel, galat luas, dan galat jumlah tetap dilaporkan sebagai outcome lain. Tidak ada penyesuaian parameter berdasarkan test set.

In [ ]:
def clean_components(binary,min_area,small_hole=18):
    # Pengganti eksplisit remove_small_objects/holes agar stabil lintas versi.
    cc=measure.label(binary,connectivity=2)
    size=np.bincount(cc.ravel())
    keep=(size>=int(min_area));keep[0]=False
    mask=keep[cc]
    bg=measure.label(~mask,connectivity=2)
    bs=np.bincount(bg.ravel())
    border=np.zeros(len(bs),dtype=bool)
    border[np.unique(np.r_[bg[0,:],bg[-1,:],bg[:,0],bg[:,-1]])]=True
    fill=(bs<=int(small_hole))&~border
    fill[0]=False
    return mask|fill[bg]

def predict_instances(sample, method='raw_otsu', *, min_area=60, thr=.64,
                      local_offset=.10, ws_distance=11):
    raw=sample['raw']
    if method=='raw_otsu':
        binary=raw < filters.threshold_otsu(raw)
    elif method=='local':
        local=filters.threshold_local(raw,block_size=49,method='gaussian',offset=local_offset)
        binary=raw < local
    elif method in {'flat_otsu','flat_fixed','flat_ws'}:
        corrected=flat_correct(sample)
        if method=='flat_otsu':
            binary=corrected<filters.threshold_otsu(corrected)
        else:
            binary=corrected<thr
    else:
        raise ValueError('Metode tidak dikenali: '+str(method))
    binary=clean_components(binary,min_area,small_hole=18)
    if method!='flat_ws':
        return measure.label(binary,connectivity=2)
    distance=ndi.distance_transform_edt(binary)
    peaks=feature.peak_local_max(ndi.gaussian_filter(distance,1),
        min_distance=int(ws_distance),threshold_abs=3.0,
        labels=binary,exclude_border=False)
    mk=np.zeros(binary.shape,dtype=bool)
    if len(peaks):mk[tuple(peaks.T)]=True
    markers=measure.label(mk)
    # Tiap komponen tanpa seed mendapat satu seed di lokasi jarak maksimum.
    cc=measure.label(binary)
    for reg in measure.regionprops(cc):
        loc=(cc==reg.label)
        if not np.any(markers[loc]):
            idx=np.argmax(np.where(loc,distance,-1))
            rr,ccol=np.unravel_index(idx,binary.shape)
            markers[rr,ccol]=markers.max()+1
    return segmentation.watershed(-distance,markers,mask=binary)

methods=['raw_otsu','local','flat_otsu','flat_fixed','flat_ws']
shown={m:predict_instances(s,m) for m in methods}
fig,ax=plt.subplots(2,3,figsize=(12,7))
lihat(ax[0,0],s['labels'],'Acuan (instance)',cmap='nipy_spectral')
for a,m in zip(ax.ravel()[1:],methods):
    lihat(a,shown[m],f'{m}: {shown[m].max()} label',cmap='nipy_spectral')
plt.tight_layout();plt.show()

### 14D. Definisi evaluasi per spesimen dan pemilihan parameter pada validasi

Pipeline untuk setiap spesimen menghasilkan label instance yang dibandingkan dengan label acuan. Precision, recall, F1 objek, IoU piksel, kesalahan jumlah, dan kesalahan area dihitung **per spesimen** terlebih dahulu. Saat dirata-ratakan, masing-masing spesimen mendapat bobot sama, sehingga satu gambar resolusi besar tidak mendominasi statistik hanya karena memiliki banyak piksel.

Untuk studi kasus ini digunakan $p_x=0.042$ mm/piksel dan $p_y=0.046$ mm/piksel sebagai angka **asumsi ilustratif**. Karena pitch dan label acuan disimulasikan, nilai MAE mm² mengukur konsistensi algoritma terhadap model sintetis, bukan ketidakpastian instrumen nyata.

Sebagai penyederhanaan yang eksplisit, *parameter grid* ditentukan sebelum hasil test dihitung. Untuk varian watershed, jarak antarpenanda merupakan parameter tambahan dan diuji pada data validasi saja.

In [ ]:
def sample_scores(pred, sample):
    ref=sample['labels']; t=ref>0;p=pred>0
    pm=pixel_metrics(p,t)
    om=instance_metrics(pred,ref,iou_threshold=.5)
    n_true=int(np.unique(ref[ref>0]).size)
    n_pred=int(np.unique(pred[pred>0]).size)
    A_true=np.count_nonzero(t)*PITCH_X_MM*PITCH_Y_MM
    A_pred=np.count_nonzero(p)*PITCH_X_MM*PITCH_Y_MM
    return {'pixel_iou':pm['iou'],'pixel_dice':pm['dice'],
            'object_f1':om['f1_obj'],'object_precision':om['precision_obj'],
            'object_recall':om['recall_obj'],
            'count_error':abs(n_true-n_pred),
            'area_mae_mm2':abs(A_true-A_pred),
            'n_true':n_true,'n_pred':n_pred,'A_true_mm2':A_true,'A_pred_mm2':A_pred}

def assess_method(sample_set, method, conf):
    return pd.DataFrame([
      dict(seed=samp['seed'],**sample_scores(predict_instances(samp,method,**conf),samp))
      for samp in sample_set])

# Grid terbatas dan eksplisit. Pilih konfigurasi terbaik dari validasi SAJA.
grids={
 'raw_otsu':[{'min_area':z} for z in [50,90,130]],
 'local':[{'min_area':z,'local_offset':off} for z in [50,90,130] for off in [.07,.10,.13]],
 'flat_otsu':[{'min_area':z} for z in [50,90,130]],
 'flat_fixed':[{'min_area':z,'thr':t} for z in [50,90,130] for t in [.57,.61,.65,.69,.73]],
 'flat_ws':[{'min_area':z,'thr':t,'ws_distance':d}
            for z in [50,90] for t in [.61,.65,.69] for d in [10,14]],
}
selected={};val_rows=[]
for method in methods:
    trials=[]
    for conf in grids[method]:
        df=assess_method(val,method,conf)
        # Objective primary F1 object, tie-breaker galat luas. Tanpa inspeksi test.
        key=(df['object_f1'].mean(),-df['area_mae_mm2'].mean())
        trials.append((key,conf,df))
    best=max(trials,key=lambda v:v[0])
    selected[method]=best[1]
    val_rows.append({'metode':method,'F1_obj_validasi':best[2]['object_f1'].mean(),
                     'IoU_pixel_validasi':best[2]['pixel_iou'].mean(),
                     'galat_luas_validasi_mm2':best[2]['area_mae_mm2'].mean(),
                     'parameter':str(best[1])})
val_summary=pd.DataFrame(val_rows).sort_values('F1_obj_validasi',ascending=False)
display(val_summary.round(3))
assert set(V_SEEDS).isdisjoint(T_SEEDS)

### 14E. Evaluasi pada test set yang disisihkan

Setelah pemilihan parameter, semua konfigurasi dibekukan. Kita menghitung metrik pada 12 spesimen uji sintetis baru. Nilai rata-rata F1 dan IoU yang dicantumkan adalah **rata-rata per-spesimen**, bukan metrik gabungan seluruh piksel. Hasilnya dapat berubah jika distribusi bentuk, iluminasi, noise, blur, kalibrasi, atau prevalensi objek berubah.

Perbandingan ini bukan uji superioritas lintas laboratorium: semua data berasal dari satu keluarga simulator, tanpa akuisisi nyata, dan model koreksi memperoleh kalibrasi sintetis yang cocok dengan data. Replikasi unit eksperimen dan ketidakpastian selanjutnya diperlakukan secara terpisah.

In [ ]:
results=[];per_specimen={}
for method in methods:
    df=assess_method(test,method,selected[method])
    per_specimen[method]=df
    results.append({'metode':method,
      'F1_obj':df['object_f1'].mean(),
      'IoU_pixel':df['pixel_iou'].mean(),
      'MAE_jumlah':df['count_error'].mean(),
      'MAE_luas_mm2':df['area_mae_mm2'].mean()})
test_summary=pd.DataFrame(results).sort_values('F1_obj',ascending=False)
display(test_summary.round(4))
fig,ax=plt.subplots(1,2,figsize=(11,3.6))
ax[0].bar(test_summary['metode'],test_summary['F1_obj'],color='0.45')
ax[0].set(title='F1 tingkat objek pada test sintetis',ylabel='F1 objek',ylim=(0,1.05))
ax[1].bar(test_summary['metode'],test_summary['MAE_luas_mm2'],color='0.6')
ax[1].set(title='Kesalahan luas proyeksi total',ylabel='MAE (mm² asumsi)')
for a in ax:a.tick_params(axis='x',rotation=30)
plt.tight_layout();plt.show()

### 14F. Confidence interval dan perbandingan berpasangan

Setiap spesimen mengalami semua metode, sehingga hasil antar-metode **berpasangan**. Untuk menggambarkan variabilitas rerata F1 (atau selisih rerata F1), lakukan bootstrap dengan menyampel ulang **identitas spesimen**, bukan piksel individual. Interval bootstrap persentil 95% di bawah adalah interval sampling bersyarat pada simulator dan kumpulan test sintetis tertentu; bukan jaminan peluang cakupan untuk populasi sensor nyata.

Ketidakpastian statistik dari pengulangan spesimen tidak mencakup semua sumber ketidakpastian metrologi. Kalibrasi pitch, anotasi, perubahan optik, serta kesalahan sistematis masih memerlukan estimasi tersendiri. Selisih positif antarmetode tidak harus berarti keuntungan yang penting secara praktis; periksa ukuran efek dan rentang intervalnya.

In [ ]:
def bootstrap_mean_ci(values,seed=100,nboot=2000):
    v=np.asarray(values,float)
    r=np.random.default_rng(seed)
    idx=r.integers(0,len(v),size=(nboot,len(v)))
    b=v[idx].mean(axis=1)
    return np.quantile(b,[.025,.975])

ci_rows=[]
for k,method in enumerate(methods):
    v=per_specimen[method]['object_f1'].to_numpy()
    lo,hi=bootstrap_mean_ci(v,seed=SEED+800+k)
    ci_rows.append({'metode':method,'rata_rata_F1':v.mean(),'CI95_bawah':lo,'CI95_atas':hi})
ci_table=pd.DataFrame(ci_rows).sort_values('rata_rata_F1',ascending=False)
display(ci_table.round(4))
# Perbandingan dua pipeline tanpa memilihnya berdasarkan hasil uji.
x=per_specimen['flat_fixed'].set_index('seed')['object_f1']
y=per_specimen['raw_otsu'].set_index('seed')['object_f1']
diff=(x-y).to_numpy()
dlo,dhi=bootstrap_mean_ci(diff,seed=SEED+901)
print(f'Selisih rata-rata F1 objek (flat_fixed - raw_otsu) = {diff.mean():.3f};'
      f' bootstrap CI95 [{dlo:.3f}, {dhi:.3f}]')
fig,ax=plt.subplots(figsize=(9.5,4))
ax.errorbar(ci_table['metode'],ci_table['rata_rata_F1'],
            yerr=[ci_table['rata_rata_F1']-ci_table['CI95_bawah'],
                  ci_table['CI95_atas']-ci_table['rata_rata_F1']],
            marker='o',ls='',capsize=4,color='0.25')
ax.set(ylim=(-.02,1.05),ylabel='Rata-rata F1 objek ± CI bootstrap 95%',
       title='Replikasi dilakukan pada tingkat spesimen (n=12)')
ax.tick_params(axis='x',rotation=25);ax.grid(alpha=.2)
plt.tight_layout();plt.show()

### 14G. Pemeriksaan kegagalan dan interpretasi geometris

Rata-rata F1 dapat menyembunyikan objek yang terlewat, *false positive*, maupun satu objek besar yang menyerap beberapa objek kecil. Karena itu, amati paling tidak satu spesimen uji dengan performa terendah dan tampilkan mask referensi serta prediksi dari parameter yang **sudah dibekukan**.

Periksa apakah kesalahan bersumber dari iluminasi yang tidak terkoreksi, blur yang mempertemukan dua objek, hasil *morphological cleanup*, pengaturan marker watershed, atau asumsi ambang. Jangan menggunakan visualisasi error pada test untuk kembali mengoptimalkan parameter lalu melaporkan test yang sama sebagai estimasi independen.

In [ ]:
# Diagnosis visual contoh terburuk untuk pipeline terbaik menurut VALIDASI.
best_method=val_summary.iloc[0]['metode']
df_best=per_specimen[best_method]
worst_seed=int(df_best.sort_values('object_f1').iloc[0]['seed'])
w=next(x for x in test if x['seed']==worst_seed)
fig,ax=plt.subplots(1,4,figsize=(14,3.4))
lihat(ax[0],w['raw'],f'Spesimen uji {worst_seed}',vmin=0,vmax=1)
lihat(ax[1],flat_correct(w),'Koreksi dark–flat',vmin=0,vmax=1)
lihat(ax[2],w['labels'],f'Ground truth: {w["labels"].max()} label',cmap='nipy_spectral')
pred=predict_instances(w,best_method,**selected[best_method])
lihat(ax[3],pred,f'Prediksi ({best_method}): {pred.max()} label',cmap='nipy_spectral')
plt.tight_layout();plt.show()
print('Ringkasan kasus:',sample_scores(pred,w))

## 15. Robustness: ketika sistem bergeser dari kondisi validasi

Parameter yang tampak berhasil pada kondisi simulasi nominal mungkin gagal pada kondisi baru. Kita memeriksa **eksperimen stres** dengan perubahan iluminasi lebih ekstrem, blur lebih lebar, dan noise lebih kuat. Dataset ini berbeda dari validasi dan test utama. Semua parameter tetap dibekukan.

Tujuannya menguji **sensitivitas bersyarat pada model sintetis**, bukan membuktikan kemampuan generalisasi industri. Dalam pengujian nyata, perubahan domain dapat berasal dari perangkat kamera, lot produksi, lokasi, preparasi spesimen, operator, atau kebiasaan anotasi. Pelaporan ideal mencakup perubahan performa dan penjelasan sumbernya.

In [ ]:
shift_set=[simulate_specimen(SEED+9000+k,shift=True) for k in range(6)]
shift_rows=[]
for method in methods:
    ss=assess_method(shift_set,method,selected[method])
    shift_rows.append({'metode':method,'F1_domain_nominal':per_specimen[method]['object_f1'].mean(),
                       'F1_domain_shift':ss['object_f1'].mean(),
                       'selisih_shift_minus_nominal':ss['object_f1'].mean()-per_specimen[method]['object_f1'].mean()})
shift_report=pd.DataFrame(shift_rows)
display(shift_report.round(3))
fig,ax=plt.subplots(1,2,figsize=(9,3.3))
lihat(ax[0],shift_set[0]['raw'],'Akuisisi domain stres',vmin=0,vmax=1)
lihat(ax[1],shift_set[0]['labels'],'Ground truth domain stres',cmap='nipy_spectral')
plt.tight_layout();plt.show()

## 16. Pengukuran ukuran tiap objek dan akibat kesalahan instance

Selain luas total, pengukuran per benda dapat memberikan distribusi ukuran. Jika dua objek saling terhubung dan sistem menghasilkan satu instance, **luas total bisa hampir benar tetapi jumlah dan distribusi luas per partikel salah besar**. Karena itu, pelaporan ilmiah sebaiknya membedakan *total area fraction*, jumlah instance, dan ukuran setiap instance yang berhasil dicocokkan dengan label acuan.

Dalam metrologi nyata, diameter ekuivalen tidak cukup untuk mengetahui ketebalan partikel, volume, ukuran pori 3D, atau massa. Hubungan ke besaran tersebut membutuhkan asumsi bentuk, geometri proyeksi, fokus, dan kalibrasi terpisah.

In [ ]:
# Distribusi luas per instance pada satu spesimen uji; tidak mengintip parameternya.
ref_area=[p.area*PITCH_X_MM*PITCH_Y_MM for p in measure.regionprops(w['labels'])]
pred_area=[p.area*PITCH_X_MM*PITCH_Y_MM for p in measure.regionprops(pred)]
fig,ax=plt.subplots(1,2,figsize=(11,3.4))
ax[0].hist(ref_area,bins=10,alpha=.65,label='Referensi')
ax[0].hist(pred_area,bins=10,alpha=.55,label='Prediksi')
ax[0].set(xlabel='Luas objek (mm², pitch asumsi)',ylabel='Jumlah instance',title='Distribusi luas')
ax[0].legend()
ax[1].scatter(np.arange(len(ref_area)),sorted(ref_area),label='Referensi',marker='o')
ax[1].scatter(np.arange(len(pred_area)),sorted(pred_area),label='Prediksi',marker='x')
ax[1].set(xlabel='Urutan ukuran',ylabel='Luas (mm², pitch asumsi)',title='Urutan area instance')
ax[1].legend();plt.tight_layout();plt.show()
print('Total area benar:',round(sum(ref_area),3),'| prediksi:',round(sum(pred_area),3),'mm²')

## 17. Trade-off kualitas–kompleksitas, dan biaya operasional

Model terbaik bergantung pada tujuan. Jika proses hanya mengukur persentase luas, label instance tidak selalu diperlukan. Jika proses menghitung cacat kritis per produk, *watershed* dan *matching* mungkin lebih penting. Jika digunakan pada kamera produksi *real time*, biaya komputasi dan stabilitas eksposur harus dipertimbangkan bersama akurasi.

Pengukuran waktu di bawah bukan benchmark lintas komputer: jalankan pada mesin yang sama, gunakan sampel sama, dan lakukan beberapa pengulangan. Nilai absolut dipengaruhi compiler pustaka, sistem operasi, beban mesin, cache, serta resolusi citra. Kualitas ilmiah juga menuntut *memory usage*, versi dependency, dan pembatasan lingkungan bila membahas skalabilitas.

In [ ]:
bench=[]
for method in methods:
    runs=[]
    for rep in range(5):
        start=time.perf_counter()
        _=predict_instances(test[0],method,**selected[method])
        runs.append((time.perf_counter()-start)*1000)
    bench.append({'metode':method,'median_ms':np.median(runs),
                  'min_ms':np.min(runs),'max_ms':np.max(runs)})
display(pd.DataFrame(bench).round(2))
print('Benchmark ini bersifat deskriptif pada lingkungan eksekusi lokal saat dijalankan.')

## 18. Verifikasi numerik dan audit reproducibility

Sebelum eksperimen dianggap siap diajarkan, periksa invarians sederhana yang tidak bergantung pada kesan visual: Otsu manual sebanding dengan pustaka, identitas Dice–IoU terpenuhi, konektivitas 4/8 berbeda sesuai definisi, luas piksel berubah kuadratis terhadap pitch isotropik, dan seluruh partisi memiliki seed berbeda.

Berikut pemeriksaan otomatis untuk mengurangi kesalahan program. Tes sintetis kecil ini **bukan pengganti** validasi empiris kamera, kalibrasi optik, evaluasi anotator manusia, atau pemeriksaan lintas versi library.

In [ ]:
# Uji unit ringkas yang dapat dijalankan setiap selesai mengedit notebook.
assert len(val)==8 and len(test)==12
assert set(V_SEEDS).isdisjoint(T_SEEDS)
assert np.all(np.isfinite(s['raw'])) and np.all((s['raw']>=0)&(s['raw']<=1))
assert np.all(np.isfinite(flat_correct(s)))
assert all(np.unique(x['labels'])[0]==0 for x in val+test)
assert all((x['raw'].shape==x['labels'].shape==x['flat'].shape) for x in val+test)
assert all(0<=results_i<=1 for results_i in test_summary['F1_obj'])
assert all(0<=results_i<=1 for results_i in test_summary['IoU_pixel'])
assert set(per_specimen.keys())==set(methods)
assert np.isclose(2*.5/(1+.5),2/3)
assert np.isclose(N*(2*px)*(2*py),4*A)
# Tidak ada rujukan ground truth saat memprediksi: prediksi tetap sama setelah labels diganti.
s0=test[0]
fake={**s0,'labels':np.zeros_like(s0['labels'])}
for method in methods:
    a=predict_instances(s0,method,**selected[method])
    b=predict_instances(fake,method,**selected[method])
    assert np.array_equal(a,b),method
print('Pemeriksaan numerik/reproducibility: LULUS.')

## 19. Audit asumsi dan batas interpretasi

Sebelum mengubah hasil simulasi menjadi klaim ilmiah atau rekomendasi alat, periksa secara kritis:

1. **Asal citra:** contoh pustaka, sintetis, atau pengukuran sensor fisik? Provenance tercatat?
2. **Measurand:** jumlah benda fisik, area proyeksi, luas permukaan 3D, atau area kode threshold?
3. **Kalibrasi:** apakah pitch pada bidang objek benar, distorsi lensa terkoreksi, dan flat/dark berlaku untuk eksposur yang diuji?
4. **Kesesuaian statistik:** apakah spesimen merupakan unit independen, atau beberapa gambar berasal dari spesimen yang sama?
5. **Definisi ground truth:** siapa anotatornya, bagaimana resolusi sengketa label, dan apakah batas fisik memang dapat dikenali dari satu citra?
6. **Kelas salah:** apakah false negative pada objek kecil lebih berbahaya dibanding false positive pada noda yang tidak berbahaya?
7. **Topologi:** apakah kesalahan terpenting merupakan hilangnya piksel, atau menyatunya dua instance akibat blur?
8. **Pilihan parameter:** apakah validasi dipisahkan dari test pada tingkat spesimen, batch, dan perangkat yang relevan?
9. **Robustness:** apakah intensitas, noise, morfologi, dan protokol akuisisi berubah dalam operasi sesungguhnya?
10. **Batas klaim:** hasil sintetis dan CI bootstrap tidak dapat digunakan sebagai bukti ketertelusuran metrologi.

**Interpretasi akhir:** angka F1 tinggi hanya berarti kesesuaian dengan ground truth menurut protokol tertentu. Keabsahan keputusan fisik dan industri memerlukan model kausal, standar acuan, kalibrasi, dan validasi eksternal.

## 20. Latihan bertingkat, HOTS, dan proyek mini

**Latihan 1 — Otsu (dasar).** Diberikan histogram 4 tingkat intensitas dengan frekuensi $[3,6,4,3]$. Hitung $\omega_0,\mu_0,\omega_1,\mu_1$, dan $\sigma_B^2$ untuk semua ambang yang membuat kedua kelas tidak kosong. Tentukan ambang terbaik dan jelaskan apakah solusi tergantung jumlah piksel secara absolut.

**Latihan 2 — Morfologi himpunan (dasar–menengah).** Untuk mask $7\times7$ dengan satu lubang dan satu piksel noise, gambar hasil erosi, dilasi, opening, dan closing dengan elemen struktur silang radius 1. Buktikan *opening* idempoten untuk contoh tersebut dan diskusikan efek *padding*.

**Latihan 3 — Ketetanggaan (menengah).** Buat mask dengan objek hanya bersentuhan diagonal. Bandingkan komponen untuk konektivitas 4 dan 8. Jelaskan kapan aturan itu dapat menyebabkan kesalahan penghitungan material berpori.

**Latihan 4 — Segmentasi lokal (menengah).** Pertahankan mask sintetis yang sama, tetapi ubah amplitudo iluminasi, PSF, dan read noise secara terpisah. Bandingkan Otsu global, ambang lokal, dan koreksi flat-field. Tampilkan kurva perubahan Dice, F1 objek, serta kesalahan luas terhadap tingkat gangguan.

**Latihan 5 — Ukuran fisik dan ketidakpastian (menengah).** Satu objek memiliki $N=3000$ piksel, $p_x=0.035$ mm/piksel, dan $p_y=0.040$ mm/piksel. Hitung luas dan diameter ekuivalen. Jika kedua pitch berkorelasi dan segmentasi memiliki ketidakpastian batas, turunkan persamaan lengkap $u_A$ dan jelaskan kapan pendekatan turunan pertama gagal.

**Latihan 6 — Matching instance (lanjutan).** Diberikan 3 mask acuan dan 4 prediksi dengan matriks IoU $3\times4$ pilihan Anda. Bandingkan hasil Hungarian matching serta keputusan pada $\tau=0.3,0.5,0.7$. Berikan contoh mengapa F1 objek berubah drastis sedangkan Dice piksel berubah sedikit.

**Latihan 7 — Sensitivitas anotasi (HOTS).** Rancang protokol anotasi dua ahli pada citra dengan batas blur. Bedakan pergeseran sistematis, variasi acak, dan perbedaan definisi objek. Usulkan statistik agreement, metode resolusi sengketa, dan pengaruh variasi anotasi terhadap validitas kesimpulan.

**Latihan 8 — Replikasi dan domain shift (HOTS).** Anda memiliki 20 batch manufaktur yang masing-masing menghasilkan 50 citra. Jelaskan mengapa split acak per-citra dapat menyebabkan leakage bila batch berbagi karakteristik. Rancang grouped split, protokol pengujian lintas batch, serta bootstrap pada unit batch dan perbandingannya dengan bootstrap per-piksel.

**Latihan 9 — Kritik rekonstruksi (HOTS).** Apakah super-resolution atau dekonvolusi selalu meningkatkan ketepatan segmentasi objek? Rumuskan skenario saat metode rekonstruksi menciptakan batas semu dan desain eksperimen yang dapat membedakan detail nyata dari artefak model.

**Proyek mini (2–3 mahasiswa):** tentukan satu tugas pengukuran (jumlah partikel, luas cacat, atau fraksi pori). Lakukan evaluasi minimal tiga pipeline dengan data berlabel dan sumber yang boleh digunakan; pisahkan validasi–uji berdasarkan unit spesimen; sertakan Dice/IoU, F1 objek, MAE besaran fisik, ketidakpastian, dan diskusi false positive/false negative. Susun laporan: perumusan masalah → model citra → ground truth → metode → eksperimen → hasil → audit kesalahan → keterbatasan → kesimpulan yang proporsional.

## 21. Rangkuman dan koneksi berikutnya

1. **Histogram** menyediakan distribusi nilai, tetapi kehilangan geometri. Ambang Otsu memaksimalkan pemisahan distribusi, bukan otomatis akurasi pengukuran.
2. **Filtering dan PSF** mengubah lebar transisi serta keterhubungan objek. Blur atau morfologi dapat menggabungkan benda fisik yang semestinya dihitung terpisah.
3. **Deteksi tepi** memberi kandidat batas, sedangkan segmentasi membentuk wilayah dan instance. Keduanya memerlukan definisi objek dan aturan keputusan eksplisit.
4. **Morfologi dan konektivitas** merupakan operasi geometris/topologis; keberhasilan menghilangkan noise tidak sama dengan mempertahankan luas atau jumlah benda yang benar.
5. **Kalibrasi** menentukan apakah keluaran disajikan sebagai piksel atau mm². Ketidakpastian pitch, noise, PSF, pilihan threshold, dan anotasi dapat memengaruhi besaran fisik secara berbeda.
6. **Evaluasi tingkat piksel dan objek** harus dipisahkan; segmentasi yang cukup baik menurut Dice bisa keliru pada jumlah partikel.
7. **Inferensi statistik** harus dilakukan pada unit replikasi yang sah dan protokol tuning yang tidak menggunakan data uji.

**Koneksi berikutnya:** Notebook 06 membahas domain frekuensi yang memperdalam relasi konvolusi–PSF–OTF/MTF, serta dampak filtering pada detail spasial; notebook tentang *inverse problems* mengkaji rekonstruksi dari data blur dan noisy; dan pengukuran ilmiah membahas kalibrasi serta propagasi ketidakpastian yang lebih menyeluruh. Dengan demikian, perjalanan dari representasi citra (01) sampai segmentasi (05) tetap berpusat pada satu gagasan: **citra adalah pengukuran, dan algoritma adalah bagian dari proses inferensi atas pengukuran tersebut.**

## 22. Referensi ilmiah dan provenance

- Otsu, N. (1979). “A Threshold Selection Method from Gray-Level Histograms.” *IEEE Transactions on Systems, Man, and Cybernetics*, 9(1), 62–66. DOI: https://doi.org/10.1109/TSMC.1979.4310076.
- Soille, P. (2003). *Morphological Image Analysis: Principles and Applications* (edisi ke-2). Springer. DOI: https://doi.org/10.1007/978-3-662-05088-0.
- Serra, J. (1982). *Image Analysis and Mathematical Morphology*. Academic Press.
- Vincent, L., & Soille, P. (1991). “Watersheds in Digital Spaces: An Efficient Algorithm Based on Immersion Simulations.” *IEEE Transactions on Pattern Analysis and Machine Intelligence*, 13(6), 583–598. DOI: https://doi.org/10.1109/34.87344.
- Szeliski, R. (2022). *Computer Vision: Algorithms and Applications* (edisi ke-2). Springer. https://szeliski.org/Book/.
- Gonzalez, R. C., & Woods, R. E. (2018). *Digital Image Processing* (edisi ke-4). Pearson.
- `scikit-image` — filtering, morphology, measurement, segmentation: https://scikit-image.org/docs/stable/api/api.html.
- `SciPy ndimage`: https://docs.scipy.org/doc/scipy/reference/ndimage.html.
- `scikit-image` example dataset: https://scikit-image.org/docs/stable/api/skimage.data.html.

**Provenance dan batas data:** hanya citra contoh `coins` berasal dari `scikit-image`. Studi kasus, citra kalibrasi, ground truth instance, variasi anotasi, skala mm/piksel, serta efek pencahayaan merupakan **data sintetis hasil generator dengan seed tetap**. Tidak tersedia dataset label manusia, pengukuran sensor terkalibrasi, bukti kesesuaian ISO, atau validasi lapangan. Dokumentasi ini dimaksudkan sebagai materi pedagogis dan eksperimen simulasi yang dapat direproduksi.